In [2]:
from sedona.spark import SedonaContext
import os

In [3]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())
sedona.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/15 11:49:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/15 11:49:11 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/08/15 11:49:11 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/08/15 11:49:11 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/08/15 11:49:11 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.geometryObjects.Geography, which is already registered.
25/08/15 11:49:11 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/08/15 11:49:11 WARN SimpleFunctionRegistry: The function st_env

# Reading Postgis table

In [11]:
table_name = "points"

postgresql_url = "jdbc:postgresql://postgis:5432/sedona"

df = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", "sedona") \
    .option("password", "postgis") \
    .option("dbtable", "points") \
    .option("driver", "org.postgresql.Driver") \
    .load()

In [12]:
df.show()

+-------+--------------------+
|   name|            location|
+-------+--------------------+
|Point A|0101000020E610000...|
|Point B|0101000020E610000...|
|Point C|0101000020E610000...|
+-------+--------------------+



In [13]:
df.selectExpr("name", "ST_GeomFromEWKB(location) AS geom").show()

+-------+-------------+
|   name|         geom|
+-------+-------------+
|Point A|POINT (10 20)|
|Point B|POINT (30 40)|
|Point C|POINT (50 60)|
+-------+-------------+



# Reading MySQL table

In [14]:
database_name = "sedona"
user_name = "sedona"
password = "sedona"

mysql_url = f"jdbc:mysql://mysql-sedona:3306/{database_name}"

df = sedona.read \
    .format("jdbc") \
    .option("url", mysql_url) \
    .option("user", user_name) \
    .option("password", password) \
    .option("dbtable", "points") \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()
df.show()

+-------+--------------------+
|   name|            location|
+-------+--------------------+
|Point A|[E6 10 00 00 01 0...|
|Point B|[E6 10 00 00 01 0...|
|Point C|[E6 10 00 00 01 0...|
+-------+--------------------+



In [15]:
df.selectExpr(
    "name",
    "ST_GeomFromMySQL(location) AS geom"
).show(3, False)

AnalysisException: [UNRESOLVED_ROUTINE] Cannot resolve function `ST_GeomFromMySQL` on search path [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].; line 1 pos 0